# EduPro — 01. Data Understanding & Exploratory Data Analysis

**Purpose:** inspect the three source tables (Users, Courses, Transactions), validate the schema, assess data quality, and identify initial learner/course patterns.

**Before running:** put the supplied Excel workbook in `Data/raw/` (or `Data/Processed/`). Run cells from top to bottom. This notebook does not modify the original workbook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT.parent
DATA_DIRS = [ROOT / "Data" / "raw"]
files = [p for folder in DATA_DIRS for pattern in ("*.xlsx", "*.xls") for p in folder.glob(pattern)]
if not files:
    raise FileNotFoundError("Place the original EduPro Excel workbook in Data/raw.")
WORKBOOK = files[0]
print("Project root:", ROOT)
print("Workbook:", WORKBOOK.name)

In [ ]:
xls = pd.ExcelFile(WORKBOOK)
print("Available sheets:", xls.sheet_names)

sheets = {name.strip().lower(): pd.read_excel(WORKBOOK, sheet_name=name) for name in xls.sheet_names}

def get_sheet(target):
    for name, frame in sheets.items():
        if name == target.lower() or target.lower() in name:
            return frame.copy()
    raise KeyError(f"Required sheet {target!r} not found. Found: {xls.sheet_names}")

users = get_sheet("Users")
courses = get_sheet("Courses")
transactions = get_sheet("Transactions")

print("Users:", users.shape)
print("Courses:", courses.shape)
print("Transactions:", transactions.shape)

In [ ]:
for title, frame in [("Users", users), ("Courses", courses), ("Transactions", transactions)]:
    print(f"\\n--- {title} ---")
    display(frame.head())
    display(pd.DataFrame({"column": frame.columns, "dtype": frame.dtypes.astype(str).values,
                          "missing": frame.isna().sum().values,
                          "missing_pct": (frame.isna().mean()*100).round(2).values}))

In [ ]:
required = {
    "Users": {"UserID", "Age", "Gender"},
    "Courses": {"CourseID", "CourseCategory", "CourseType", "CourseLevel", "CourseRating"},
    "Transactions": {"UserID", "CourseID", "TransactionDate", "Amount"},
}
frames = {"Users":users, "Courses":courses, "Transactions":transactions}
for name, cols in required.items():
    missing = cols - set(frames[name].columns)
    print(name, "missing required columns:", sorted(missing))
    assert not missing, f"{name} is missing required columns: {missing}"

print("Required schema validation passed.")

In [ ]:
for title, frame, key in [
    ("Users", users, "UserID"),
    ("Courses", courses, "CourseID"),
    ("Transactions", transactions, None)
]:
    print(f"{title}: duplicate rows =", int(frame.duplicated().sum()))
    if key:
        print(f"{title}: duplicate {key} =", int(frame[key].duplicated().sum()))
print("Transaction date parse preview:")
display(pd.to_datetime(transactions["TransactionDate"], errors="coerce").describe())

In [ ]:
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
courses["CourseRating"] = pd.to_numeric(courses["CourseRating"], errors="coerce")
transactions["Amount"] = pd.to_numeric(transactions["Amount"], errors="coerce")
transactions["TransactionDate"] = pd.to_datetime(transactions["TransactionDate"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
users["Age"].dropna().plot(kind="hist", bins=20, ax=axes[0], title="Learner age distribution")
courses["CourseRating"].dropna().plot(kind="hist", bins=15, ax=axes[1], title="Course rating distribution")
plt.tight_layout()
plt.show()

In [ ]:
category_counts = courses["CourseCategory"].fillna("Unknown").value_counts().head(15)
display(category_counts.rename_axis("Course category").to_frame("Course count"))

level_counts = courses["CourseLevel"].fillna("Unknown").value_counts()
display(level_counts.rename_axis("Course level").to_frame("Course count"))

monthly = transactions.copy()
monthly["TransactionDate"] = pd.to_datetime(monthly["TransactionDate"], errors="coerce")
monthly = monthly.dropna(subset=["TransactionDate"])
monthly["Month"] = monthly["TransactionDate"].dt.to_period("M").astype(str)
monthly_counts = monthly.groupby("Month").size()
monthly_counts.plot(kind="line", marker="o", figsize=(11,4), title="Monthly transaction/enrollment trend")
plt.ylabel("Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## EDA checkpoint

Before continuing, review:

- Required columns are present in all three tables.
- Duplicate IDs and missing values are understood.
- Age, rating, amount and transaction date distributions are reasonable.
- Category and level values are consistent (for example, spelling variants should be investigated).
- Transaction records refer to valid UserIDs and CourseIDs.

Do not delete rows only to make charts look cleaner. Record any cleaning decision in the research paper.